# ML-07 — Baseline Action Score and Top-10 Review

**Lane 4: CTR / Engagement Opportunity Scoring**

Skills loaded: `building-baselines/SKILL.md` + `flyrank/flyrank-data/SKILL.md`

All claims use careful, observed language (observational / measured / directional / decision-support).  
No client names, domains, URLs, or private queries appear anywhere in this notebook.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

---
## 1. Two Signal Checks and My Rule's Reasoning

### My rule in plain words

A page is worth a CTR review if it is **visible in search** (many impressions) but **under-captures clicks relative to other pages at the same search depth**. A large CTR gap multiplied by high impression volume = the most actionable opportunity. Pages that are also stale (not updated in 180+ days) get the more aggressive action label.

**Reason codes this rule outputs:**
- `stale_low_ctr` — below-tier CTR AND stale content (≥180 days since last update)
- `large_ctr_gap` — CTR is at least 50% below the tier median
- `moderate_ctr_gap` — CTR is below tier median but by less than 50%
- `ctr_at_or_above_tier` — page's CTR meets or beats its position-tier peers → monitor only

**Action labels:**
- `refresh_and_optimize_cta` — rewrite + title/meta optimization needed
- `optimize_title_and_meta` — title/meta/snippet optimization alone
- `monitor` — no immediate action

---

### Signal 1 — CTR varies by position tier (behind the CTR-fix flag)

The FlyRank CTR-fix logic flags pages whose CTR falls below peers at similar search positions. The signal that flag leans on: **does CTR actually differ across position tiers?** If CTR is flat across tiers, position-adjusted comparison is meaningless.

**Hypothesis:** median CTR should be highest in `top_3` and fall monotonically toward `deep`.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(DATA_PATH)

# Signal 1: CTR by position tier
# Filter: must have valid position data and meaningful impressions
s1 = df[
    (df['avg_position'] > 0) &           # 0 = no position data (1,205 rows)
    (df['impressions_90d'] >= 100) &      # floor: pages with almost no visibility are noise
    (df['position_tier'] != 'no_data')
].copy()

TIER_ORDER = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']

tbl1 = (
    s1.groupby('position_tier')
    .agg(
        n          = ('ctr', 'count'),
        median_ctr = ('ctr', 'median'),
        p25_ctr    = ('ctr', lambda x: x.quantile(0.25)),
        p75_ctr    = ('ctr', lambda x: x.quantile(0.75)),
    )
    .reindex(TIER_ORDER)
    .round(3)
)

print('Signal 1 — CTR by position tier')
print("Filter: avg_position > 0 AND impressions_90d >= 100 AND position_tier != 'no_data'")
print(f'n = {len(s1):,}')
print()
print(tbl1.to_string())
print()
print('VERDICT: CONFIRMED')
print('Median CTR falls from 0.23% (page_1) to 0.00% (deep). The striking zone')
print('(positions 11-20) already shows a sharp drop to 0.15%, and page_3_5 halves')
print('again to 0.06%. Position-tier-adjusted CTR comparison is a meaningful signal.')
print()
print('Note: ctr values are x100 percentages (0.23 = 0.23%, NOT 23%).')
print('Note: top_3 median (0.19) is slightly below page_1 (0.23) -- navigational')
print('      queries with low click-intent inflate top_3 impressions without clicks.')

Signal 1 — CTR by position tier
Filter: avg_position > 0 AND impressions_90d >= 100 AND position_tier != 'no_data'
n = 22,006

                  n  median_ctr  p25_ctr  p75_ctr
position_tier                                    
top_3           533        0.19     0.05     0.48
page_1         8633        0.23     0.09     0.46
striking       5903        0.15     0.00     0.34
page_3_5       6058        0.06     0.00     0.19
deep            879        0.00     0.00     0.00

VERDICT: CONFIRMED
Median CTR falls from 0.23% (page_1) to 0.00% (deep). The striking zone
(positions 11-20) already shows a sharp drop to 0.15%, and page_3_5 halves
again to 0.06%. Position-tier-adjusted CTR comparison is a meaningful signal.

Note: ctr values are x100 percentages (0.23 = 0.23%, NOT 23%).
Note: top_3 median (0.19) is slightly below page_1 (0.23) -- navigational
      queries with low click-intent inflate top_3 impressions without clicks.


---

### Signal 2 — Staleness is linked to lower impression volume (behind the refresh flags)

FlyRank's content-refresh flags use staleness (days since last update) as a trigger. The signal behind that logic: **do stale pages actually accumulate fewer impressions, suggesting they are losing ground in search?** If stale pages perform equally well, staleness is not a useful refresh trigger.

**Hypothesis:** pages in the `181+` freshness tier should have lower median impressions than recently-updated pages.

In [2]:
# Signal 2: Impressions and decline rate by freshness tier
FRESH_ORDER = ['0-30', '31-90', '91-180', '181+']

s2 = df[df['freshness_tier'].isin(FRESH_ORDER)].copy()

tbl2 = (
    s2.groupby('freshness_tier')
    .agg(
        n                  = ('impressions_90d', 'count'),
        median_impressions  = ('impressions_90d', 'median'),
        pct_declining       = ('trend_direction', lambda x: (x == 'down').mean() * 100),
    )
    .reindex(FRESH_ORDER)
    .round(1)
)

print('Signal 2 -- Impressions by freshness tier (days_since_last_update)')
print(f'n = {len(s2):,}')
print()
print(tbl2.to_string())
print()
print('VERDICT: MIXED')
print('The 91-180 tier has the HIGHEST median impressions (1,692) -- likely because')
print('pages updated 3-6 months ago have had time to accumulate search history')
print('but are not yet degraded. The 181+ tier drops sharply to 15.5 median')
print('impressions (n=174 only -- most very-stale pages have fallen off search,')
print('suggesting survival bias in the dataset).')
print()
print('Implication: staleness alone is not a clean trigger. Pages that survived')
print('181+ days stale tend to be very low-impression (little to gain from refreshing).')
print('The rule correctly combines staleness WITH a CTR gap so that stale-but-invisible')
print('pages do not crowd the top of the queue.')

Signal 2 -- Impressions by freshness tier (days_since_last_update)
n = 30,000

                    n  median_impressions  pct_declining
freshness_tier                                          
0-30            20480               470.0           51.1
31-90             175               510.0           58.9
91-180           9171              1692.0           61.1
181+              174                15.5           47.1

VERDICT: MIXED
The 91-180 tier has the HIGHEST median impressions (1,692) -- likely because
pages updated 3-6 months ago have had time to accumulate search history
but are not yet degraded. The 181+ tier drops sharply to 15.5 median
impressions (n=174 only -- most very-stale pages have fallen off search,
suggesting survival bias in the dataset).

Implication: staleness alone is not a clean trigger. Pages that survived
181+ days stale tend to be very low-impression (little to gain from refreshing).
The rule correctly combines staleness WITH a CTR gap so that stale-but-invi

---
## 2. Build the Ranked Queue (writes the CSV)

### The scoring rule

```
tier_median_ctr   = median CTR of all pages in the same position_tier
ctr_gap           = tier_median_ctr - page_ctr    (positive = underperforming)
score             = ctr_gap x log1p(impressions_90d)
```

**Why this formula?**  
- `ctr_gap` is the opportunity: how far below its peer group is this page?  
- `log1p(impressions_90d)` is the leverage: a large gap on 1 impression is worthless; the same gap on 200,000 impressions is a priority fix.  
- No fitted weights — fully transparent and human-readable.

**No leakage:** `trend_direction`, `trend_pct`, and `is_declining_label` are NOT used.  
Features used: `ctr`, `avg_position`, `position_tier`, `impressions_90d`, `days_since_last_update`.

In [3]:
import os
import json

# --- Build the scored working set ---
# Same filter as Signal 1: valid position data, >=100 impressions, known tier
working = df[
    (df['avg_position'] > 0) &
    (df['impressions_90d'] >= 100) &
    (df['position_tier'] != 'no_data')
].copy()

# Step 1: compute tier median CTR (the peer benchmark)
tier_median_ctr = working.groupby('position_tier')['ctr'].median()
print('Tier median CTRs (x100 percentages):')
for tier in TIER_ORDER:
    if tier in tier_median_ctr.index:
        print(f'  {tier:<10}: {tier_median_ctr[tier]:.3f}')
print()

working['tier_median_ctr'] = working['position_tier'].map(tier_median_ctr)

# Step 2: CTR gap (positive = page is below its tier median = opportunity)
working['ctr_gap'] = working['tier_median_ctr'] - working['ctr']

# Step 3: score = gap x log-volume (no fitted parameters)
working['score'] = working['ctr_gap'] * np.log1p(working['impressions_90d'])

# Step 4: ONE reason code per row
def assign_reason(row):
    if row['score'] > 0 and row['days_since_last_update'] >= 180:
        return 'stale_low_ctr'
    elif row['score'] > 0 and row['ctr_gap'] >= row['tier_median_ctr'] * 0.5:
        return 'large_ctr_gap'
    elif row['score'] > 0:
        return 'moderate_ctr_gap'
    else:
        return 'ctr_at_or_above_tier'

# Step 5: action label
def assign_action(row):
    if row['score'] > 0 and row['days_since_last_update'] >= 180:
        return 'refresh_and_optimize_cta'
    elif row['score'] > 0:
        return 'optimize_title_and_meta'
    else:
        return 'monitor'

working['reason_code'] = working.apply(assign_reason, axis=1)
working['action']      = working.apply(assign_action, axis=1)

# Step 6: rank by score descending
ranked = working.sort_values('score', ascending=False).reset_index(drop=True)
ranked['rank'] = ranked.index + 1

n_actionable = (ranked['score'] > 0).sum()
print(f'Scored rows  : {len(ranked):,}')
print(f'Score range  : {ranked["score"].min():.2f} to {ranked["score"].max():.2f}')
print(f'Rows with score > 0 (actionable): {n_actionable:,} ({n_actionable/len(ranked)*100:.1f}%)')
print()
print('Action distribution:')
for act, cnt in ranked['action'].value_counts().items():
    print(f'  {act:<26}: {cnt:>6,}')
print()
print('Reason code distribution:')
for rc, cnt in ranked['reason_code'].value_counts().items():
    print(f'  {rc:<24}: {cnt:>6,}')
print()

# Step 7: write ranked queue to CSV
OUT_COLS = [
    'rank', 'content_id', 'client_id', 'position_tier', 'avg_position',
    'ctr', 'tier_median_ctr', 'ctr_gap', 'impressions_90d',
    'days_since_last_update', 'score', 'reason_code', 'action'
]
os.makedirs('../../work/outputs', exist_ok=True)
OUTPUT_PATH = '../../work/outputs/baseline_action_score.csv'
ranked[OUT_COLS].to_csv(OUTPUT_PATH, index=False)
print(f'CSV written to: work/outputs/baseline_action_score.csv')
print(f'Rows in CSV   : {len(ranked):,}')

# Save metrics receipt (committable -- small JSON, no PII, no raw data)
metrics = {
    'n_scored': int(len(ranked)),
    'n_actionable': int(n_actionable),
    'score_min': round(float(ranked['score'].min()), 4),
    'score_max': round(float(ranked['score'].max()), 4),
    'action_counts': ranked['action'].value_counts().to_dict(),
    'reason_code_counts': ranked['reason_code'].value_counts().to_dict(),
    'tier_median_ctr': {k: round(float(v), 4) for k, v in tier_median_ctr.items()},
}
with open('../../work/outputs/w04_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics JSON written: work/outputs/w04_metrics.json')

Tier median CTRs (x100 percentages):
  top_3     : 0.190
  page_1    : 0.230
  striking  : 0.150
  page_3_5  : 0.060
  deep      : 0.000



Scored rows  : 22,006
Score range  : -59.47 to 2.82
Rows with score > 0 (actionable): 10,307 (46.8%)

Action distribution:
  monitor                   : 11,699
  optimize_title_and_meta   : 10,295
  refresh_and_optimize_cta  :     12

Reason code distribution:
  ctr_at_or_above_tier    : 11,699
  large_ctr_gap           :  7,376
  moderate_ctr_gap        :  2,919
  stale_low_ctr           :     12



CSV written to: work/outputs/baseline_action_score.csv
Rows in CSV   : 22,006
Metrics JSON written: work/outputs/w04_metrics.json


---
## 3. Top-10 Review

For each of the top 10: **the action, why it is there, and what would make it wrong.**

In [4]:
top10_display = ranked.head(10)[[
    'rank', 'position_tier', 'ctr', 'tier_median_ctr', 'ctr_gap',
    'impressions_90d', 'days_since_last_update', 'score', 'reason_code', 'action'
]].copy()
top10_display.columns = [
    'rank', 'position_tier', 'ctr', 'tier_med', 'ctr_gap',
    'impressions_90d', 'days_upd', 'score', 'reason_code', 'action'
]
top10_display = top10_display.round({'ctr': 3, 'tier_med': 3, 'ctr_gap': 3, 'score': 3})
print('Top-10 rows\n')
print(top10_display.to_string(index=False))

Top-10 rows

 rank position_tier  ctr  tier_med  ctr_gap  impressions_90d  days_upd  score   reason_code                  action
    1        page_1 0.00      0.23     0.23           208678       104  2.817 large_ctr_gap optimize_title_and_meta
    2        page_1 0.01      0.23     0.22           140079        20  2.607 large_ctr_gap optimize_title_and_meta
    3        page_1 0.01      0.23     0.22           112434        20  2.559 large_ctr_gap optimize_title_and_meta
    4        page_1 0.03      0.23     0.20           223271        20  2.463 large_ctr_gap optimize_title_and_meta
    5        page_1 0.02      0.23     0.21           119217         7  2.455 large_ctr_gap optimize_title_and_meta
    6        page_1 0.01      0.23     0.22            65138        26  2.439 large_ctr_gap optimize_title_and_meta
    7        page_1 0.01      0.23     0.22            56363        20  2.407 large_ctr_gap optimize_title_and_meta
    8        page_1 0.01      0.23     0.22            4687

### Top-10 one-line review

| Rank | Action | Why it's there | What would make it wrong |
|------|--------|----------------|---------------------------|
| 1 | optimize_title_and_meta | page_1 placement, 208k impressions, CTR=0.00% vs 0.23% tier median — massive wasted visibility | CTR is 0.00% because it serves navigational brand queries (users don't click — they already know the brand); title optimization won't help |
| 2 | optimize_title_and_meta | 140k impressions, CTR=0.01%, page_1 — near-zero click capture on high-volume queries | The page is deliberately no-click content (e.g. a definition snippet that fully answers inline); low CTR is the correct outcome for this intent |
| 3 | optimize_title_and_meta | 112k impressions, CTR=0.01%, page_1 — same pattern as rank 2, huge reach / tiny CTR | If queries driving this page are predominantly informational zero-click, optimizing the title will have no meaningful effect on clicks |
| 4 | optimize_title_and_meta | 223k impressions (highest in top-10), CTR=0.03%, page_1 — largest absolute impression pool at page depth | Volume could be driven by broad head terms where no single result gets high CTR; opportunity may be structural, not a title/meta problem |
| 5 | optimize_title_and_meta | 119k impressions, CTR=0.02%, updated only 7 days ago — recently refreshed but still below tier | If the recent refresh already addressed CTR, the signal may not yet have propagated into GSC data; checking in 30 days may be more appropriate |
| 6 | optimize_title_and_meta | 65k impressions, CTR=0.01%, page_1 — strong gap relative to peers | The page may rank for queries where a rich SERP feature (FAQ, video, shopping) captures all clicks before the organic result; title changes cannot compete with that |
| 7 | optimize_title_and_meta | 56k impressions, CTR=0.01%, page_1 — consistent low-CTR pattern | If this content serves brand-awareness or top-of-funnel traffic that converts through other paths (direct, email), low CTR may be an intentional outcome |
| 8 | optimize_title_and_meta | 47k impressions, CTR=0.01%, updated 15 days ago — fresh content still underperforming | A very recent title change may not have re-indexed yet; CTR measurement lags behind; a 30-day re-check avoids acting on stale signal |
| 9 | optimize_title_and_meta | 134k impressions, CTR=0.03%, not updated in 104 days — large volume with a stale signal | If position has been stable for 104 days despite low CTR, the market may have a structurally low CTR for this topic and the peer benchmark pools different intents |
| 10 | optimize_title_and_meta | 73k impressions, CTR=0.02%, page_1 | The page_1 median (0.23%) pools very different query types; if this page's query mix skews informational-heavy, a lower CTR is structurally correct and unbeatable by optimization |

---
## 4. Weak Picks + Leakage Check

### Weak picks identified in the top-10

**Pattern weakness — all top-10 are `page_1` with large impression volumes and near-zero CTR.**

This is structurally driven by the formula: `ctr_gap x log1p(impressions)`. Pages at `page_1` depth have the largest tier median CTR (0.23%), so any page with near-zero CTR gets the maximum gap. When multiplied by 100k–200k impressions, the score dominates the ranking.

**Risk:** near-zero CTR at high impression volume likely reflects pages ranking for queries where clicks are structurally suppressed — knowledge panels, AI overviews, navigational intent, or brand queries. Title optimization cannot fix these.

**Mitigation for a next iteration:** exclude `navigational` `main_intent` from the scored pool, or require `clicks_90d >= 5` as a floor so completely zero-click pages (which may be serving correctly) do not dominate the queue.

In [5]:
# --- Weak pick analysis ---
print('=== Weak pick analysis ===')
print()

top10_full = ranked.head(10)
intent_dist = top10_full['main_intent'].fillna('unknown').value_counts()
print('Top-10 intent distribution:')
for intent, cnt in intent_dist.items():
    print(f'  {intent:<16}: {cnt}')

zero_click = (top10_full['clicks_90d'] == 0).sum()
low_click  = (top10_full['clicks_90d'] < 5).sum()
print(f'\nTop-10 zero-click pages (clicks_90d == 0): {zero_click} of 10')
print(f'Top-10 very low click pages (clicks_90d < 5): {low_click} of 10')

# --- Leakage check ---
print()
print('=== Leakage check ===')
FEATURES_USED = ['ctr', 'avg_position', 'position_tier', 'impressions_90d', 'days_since_last_update']
FORBIDDEN     = ['trend_direction', 'trend_pct']

print('Features used in the score:')
notes = {
    'ctr'                    : 'observed 90d rate -- defines ctr_gap, NOT the model label',
    'avg_position'           : 'observed GSC search position',
    'position_tier'          : 'bucketed from avg_position',
    'impressions_90d'        : 'trailing 90d impression volume',
    'days_since_last_update' : 'content metadata (age since last edit)',
}
for col in FEATURES_USED:
    print(f'  {col:<24} -- {notes[col]}')

print()
print('Forbidden columns verified NOT used:')
for col in FORBIDDEN:
    print(f'  {col:<16}: not present in score formula -- CLEAN')

print()
print('LEAKAGE CHECK: PASSED')
print('No future-window or label-derived inputs in the score formula.')
print()
print('Note: ctr defines the scoring target (ctr_gap). In the model step (ML-08),')
print('ctr itself will be excluded from the feature set since it constructs the label.')

=== Weak pick analysis ===

Top-10 intent distribution:
  informational   : 7
  transactional   : 2
  commercial      : 1

Top-10 zero-click pages (clicks_90d == 0): 1 of 10
Top-10 very low click pages (clicks_90d < 5): 2 of 10

=== Leakage check ===
Features used in the score:
  ctr                      -- observed 90d rate -- defines ctr_gap, NOT the model label
  avg_position             -- observed GSC search position
  position_tier            -- bucketed from avg_position
  impressions_90d          -- trailing 90d impression volume
  days_since_last_update   -- content metadata (age since last edit)

Forbidden columns verified NOT used:
  trend_direction : not present in score formula -- CLEAN
  trend_pct       : not present in score formula -- CLEAN

LEAKAGE CHECK: PASSED
No future-window or label-derived inputs in the score formula.

Note: ctr defines the scoring target (ctr_gap). In the model step (ML-08),
ctr itself will be excluded from the feature set since it constructs th

---
## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal verdicts with visible bucket tables and n: CONFIRMED (Signal 1), MIXED (Signal 2)
- [x] At least one signal is flag-linked: Signal 1 is behind the CTR-fix flag; Signal 2 behind refresh flags
- [x] One rule with a score, ONE reason code per row, and an action label
- [x] Ranked queue written from notebook to work/outputs/baseline_action_score.csv
- [x] Top-10 reviewed with what-would-make-it-wrong for each of the 10
- [x] No future-window or label-derived inputs: leakage check passed
- [x] Committed to repo under work/notebooks/ -- then submit your repo URL on the card. Done.